# UPI Fraud Detection Analysis
### Using Pandas, NumPy, Matplotlib & Seaborn

**Dataset:** UPI Transactions 2024 (Kaggle) — 250,000 real UPI transaction records

**Goal:** Explore transaction patterns and understand where/when/how fraud occurs

---
## Step 1: Import Libraries

In [3]:
# import pandas -> used to load and work with the data in table (DataFrame) form
import pandas as pd

# import numpy -> used for numerical/array operations (mean, median, log, etc.)
import numpy as np

# import matplotlib.pyplot -> used to draw charts and graphs
import matplotlib.pyplot as plt

# import seaborn -> built on top of matplotlib, gives nicer looking statistical charts
import seaborn as sns

# set a clean dark-grid style for all our seaborn plots
sns.set_style("darkgrid")

# set a default figure size so every plot has a similar look
plt.rcParams["figure.figsize"] = (8,5)

# ignore warning messages so the notebook output stays clean and readable
import warnings
warnings.filterwarnings("ignore")

# just a confirmation message so I know this cell ran fine
print("Libraries imported successfully")

Libraries imported successfully


## Step 2: Load the Dataset

In [4]:
# read the csv file from the same folder and store it in a DataFrame called df
df = pd.read_csv("upi_transactions_2024.csv")

# .head() shows only the first 5 rows -> quick check that the data loaded correctly
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'upi_transactions_2024.csv'

## Step 3: Check the Shape of the Data

In [ ]:
# df.shape returns a tuple (number_of_rows, number_of_columns)
# shape[0] = rows, shape[1] = columns
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

## Step 4: Dataset Info

In [ ]:
# .info() prints column names, their data types, and how many non-null values each has
# helpful to quickly spot columns that might need type conversion or cleaning
df.info()

## Step 5: Statistical Summary

In [ ]:
# .describe() gives count, mean, std, min, 25%, 50%, 75%, max for numeric columns
# useful to spot outliers, e.g. an unusually large transaction amount
df.describe()

## Step 6: Check for Missing Values

In [ ]:
# .isnull() marks every missing value as True, .sum() then counts True values per column
# if any column shows a number > 0 here, that column has missing data to handle
df.isnull().sum()

## Step 7: Check & Remove Duplicates

In [ ]:
# .duplicated() flags rows that are exact copies of an earlier row
# .sum() counts how many such duplicate rows exist
print("Duplicate rows before cleaning:", df.duplicated().sum())

# drop_duplicates() removes those repeated rows, keeping only the first occurrence
df = df.drop_duplicates()

# print the new shape so I can confirm rows were (or weren't) removed
print("Shape after removing duplicates:", df.shape)

## Step 8: Clean Up Columns (dates & names)

In [ ]:
# convert the timestamp column (currently plain text) into an actual datetime type
# this lets pandas understand it as a date/time instead of a string
df["timestamp"] = pd.to_datetime(df["timestamp"])

# print the earliest and latest date to confirm the full date range of the data
print("Data spans from", df["timestamp"].min(), "to", df["timestamp"].max())

# rename a few columns that have spaces in their names -> easier to type later (df.amount_inr
# instead of df["amount (INR)"])
df.rename(columns={"transaction id":"transaction_id",
                    "transaction type":"transaction_type",
                    "amount (INR)":"amount_inr"}, inplace=True)

# print all column names to double check the renaming worked
df.columns.tolist()

## Step 9: Fraud vs Genuine — Class Distribution

In [ ]:
# fraud_flag column: 0 means genuine transaction, 1 means fraud
# value_counts() counts how many rows fall into each category
fraud_counts = df["fraud_flag"].value_counts()
print(fraud_counts)

# normalize=True converts counts into proportions (0 to 1), *100 turns that into a percentage
fraud_percent = df["fraud_flag"].value_counts(normalize=True) * 100
print("\nPercentage:\n", fraud_percent.round(3))

## Step 10: Visualize the Class Imbalance

In [ ]:
# fraud is only ~0.19% of the data, so on a normal scale the fraud bar would be invisible
# plt.figure() opens a new blank figure with the given size (width, height in inches)
plt.figure(figsize=(6,5))

# sns.countplot() automatically counts how many rows belong to each fraud_flag value and
# draws a bar for each -> no need to compute the counts ourselves first
ax = sns.countplot(x="fraud_flag", data=df, hue="fraud_flag",
                    palette=["#2ecc71","#e74c3c"], legend=False)

# force the y-axis onto a log scale so both the huge genuine bar and tiny fraud bar are visible
plt.yscale("log")

# add title and axis labels so the chart is self-explanatory
plt.title("Fraud vs Genuine Transactions (log scale)", fontsize=14)
plt.xlabel("Fraud Flag (0 = Genuine, 1 = Fraud)")
plt.ylabel("Number of Transactions (log scale)")

# loop through every bar drawn and print its exact count on top of it
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x()+p.get_width()/2, p.get_height()),
                ha="center", va="bottom", fontsize=11)

# actually display the chart
plt.show()

## Step 11: Fraud Rate by Transaction Type

In [ ]:
# groupby("transaction_type") splits the data into groups (P2P, P2M, Bill Payment, Recharge)
# ["fraud_flag"].mean() then finds the average of 0s and 1s within each group
# since fraud_flag is 0/1, the average IS the fraud rate (e.g. 0.05 = 5% fraud)
type_fraud = df.groupby("transaction_type")["fraud_flag"].mean().sort_values(ascending=False) * 100
print("Fraud rate (%) by transaction type:\n")
print(type_fraud.round(3))

# plot this Series directly as a bar chart -> pandas Series has a built-in .plot() shortcut
plt.figure(figsize=(8,5))
type_fraud.plot(kind="bar", color="tomato")
plt.title("Fraud Rate (%) by Transaction Type")
plt.ylabel("Fraud Rate (%)")
plt.xticks(rotation=0)   # keep the category labels horizontal, easier to read
plt.tight_layout()       # auto-adjusts spacing so labels don't get cut off
plt.show()

## Step 12: Fraud Rate by Merchant Category

In [ ]:
# same grouping idea as the previous cell, just applied to merchant_category this time
cat_fraud = df.groupby("merchant_category")["fraud_flag"].mean().sort_values(ascending=False) * 100
print("Fraud rate (%) by merchant category:\n")
print(cat_fraud.round(3))

plt.figure(figsize=(10,6))
cat_fraud.plot(kind="bar", color="#c0392b")
plt.title("Fraud Rate (%) by Merchant Category")
plt.ylabel("Fraud Rate (%)")
plt.xticks(rotation=45)  # rotate labels 45 degrees since category names are longer here
plt.tight_layout()
plt.show()

## Step 13: Transaction Amount — Fraud vs Genuine (Boxplot)

In [ ]:
# a boxplot shows the median, quartiles, and outliers of a numeric column, split by group
plt.figure(figsize=(8,5))
sns.boxplot(x="fraud_flag", y="amount_inr", data=df, hue="fraud_flag",
            palette=["#2ecc71","#e74c3c"], legend=False)

# transaction amounts are heavily skewed (a few very large ones), so use a log y-axis
# to make the box and whiskers readable instead of squashed near zero
plt.yscale("log")
plt.title("Transaction Amount (log scale): Fraud vs Genuine")
plt.xlabel("Fraud Flag (0 = Genuine, 1 = Fraud)")
plt.ylabel("Amount INR (log scale)")
plt.show()

## Step 14: Average & Median Amount Comparison

In [ ]:
# compare the average (mean) transaction amount between fraud and genuine groups
avg_amount = df.groupby("fraud_flag")["amount_inr"].mean()

# also compare the median, which is less affected by extreme outlier values than the mean
median_amount = df.groupby("fraud_flag")["amount_inr"].median()

print("Average amount:\n", avg_amount.round(2))
print("\nMedian amount:\n", median_amount)

## Step 15: Fraud Rate by Hour of Day

In [ ]:
# group by the hour_of_day column (0 to 23) and get the fraud rate for each hour
hourly_fraud = df.groupby("hour_of_day")["fraud_flag"].mean() * 100

plt.figure(figsize=(10,5))
# a line plot works well here since hour of day is a continuous/sequential value
sns.lineplot(x=hourly_fraud.index, y=hourly_fraud.values, marker="o", color="purple")
plt.title("Fraud Rate (%) by Hour of the Day")
plt.xlabel("Hour of Day (0-23)")
plt.ylabel("Fraud Rate (%)")
plt.xticks(range(0,24))   # force every hour 0-23 to show on the x-axis
plt.grid(True)
plt.show()

## Step 16: Fraud Rate by Day of Week

In [ ]:
# define the correct Monday->Sunday order, otherwise pandas would sort the days alphabetically
day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]

# group by day_of_week, get fraud rate, then reindex() to force our custom day order
day_fraud = df.groupby("day_of_week")["fraud_flag"].mean().reindex(day_order) * 100

plt.figure(figsize=(9,5))
sns.barplot(x=day_fraud.index, y=day_fraud.values, hue=day_fraud.index,
            palette="viridis", legend=False)
plt.title("Fraud Rate (%) by Day of Week")
plt.ylabel("Fraud Rate (%)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Step 17: Fraud Rate by Device Type & Network Type

In [5]:
# two separate groupings: one by device_type, one by network_type
device_fraud = df.groupby("device_type")["fraud_flag"].mean() * 100
network_fraud = df.groupby("network_type")["fraud_flag"].mean() * 100

# plt.subplots(1,2, ...) creates one row of 2 side-by-side charts sharing one figure
fig, axes = plt.subplots(1, 2, figsize=(12,5))

# plot the first chart onto axes[0]
device_fraud.sort_values(ascending=False).plot(kind="bar", ax=axes[0], color="#2980b9")
axes[0].set_title("Fraud Rate (%) by Device Type")
axes[0].set_ylabel("Fraud Rate (%)")
axes[0].tick_params(axis="x", rotation=0)

# plot the second chart onto axes[1]
network_fraud.sort_values(ascending=False).plot(kind="bar", ax=axes[1], color="#8e44ad")
axes[1].set_title("Fraud Rate (%) by Network Type")
axes[1].set_ylabel("Fraud Rate (%)")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

print("Fraud rate by device:\n", device_fraud.round(3))
print("\nFraud rate by network:\n", network_fraud.round(3))

NameError: name 'df' is not defined

## Step 18: Top 10 States by Fraud Count

In [ ]:
# first filter the DataFrame to keep ONLY fraudulent rows (fraud_flag == 1)
# then count how many times each state appears using value_counts()
# .head(10) keeps just the top 10 states with the most fraud cases
state_fraud = df[df["fraud_flag"]==1]["sender_state"].value_counts().head(10)

plt.figure(figsize=(10,6))
# a horizontal bar chart (y=state names) is easier to read when labels are long
sns.barplot(x=state_fraud.values, y=state_fraud.index, hue=state_fraud.index,
            palette="Reds_r", legend=False)
plt.title("Top 10 States by Number of Fraud Transactions")
plt.xlabel("Number of Fraud Transactions")
plt.ylabel("Sender State")
plt.show()

## Step 19: Fraud Rate by Sender Bank

In [ ]:
# group by sender_bank and calculate fraud rate for each of the 8 banks in the data
bank_fraud = df.groupby("sender_bank")["fraud_flag"].mean().sort_values(ascending=False) * 100

plt.figure(figsize=(9,5))
bank_fraud.plot(kind="bar", color="#16a085")
plt.title("Fraud Rate (%) by Sender Bank")
plt.ylabel("Fraud Rate (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Step 20: Does Transaction Status Relate to Fraud?

In [ ]:
# pd.crosstab() builds a cross-tabulation table between two columns
# normalize="index" makes each ROW sum to 100%, so we get the fraud % WITHIN each status
status_fraud = pd.crosstab(df["transaction_status"], df["fraud_flag"], normalize="index") * 100
print("Fraud rate (%) within each transaction status:\n")
print(status_fraud.round(3))

## Step 21: Correlation Heatmap

In [ ]:
# select only the numeric columns that make sense to correlate against fraud_flag
plt.figure(figsize=(7,5))
numeric_df = df[["amount_inr","hour_of_day","is_weekend","fraud_flag"]]

# .corr() computes the correlation coefficient between every pair of numeric columns
corr = numeric_df.corr()

# sns.heatmap() draws this correlation matrix as a color-coded grid
# annot=True prints the actual numbers inside each cell, fmt controls decimal places
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".3f")
plt.title("Correlation Heatmap of Numeric Features")
plt.show()

---
## Summary of Findings

- Fraud is extremely rare in this dataset: only **0.19%** of the 2,50,000 transactions are fraudulent.
- Fraud rate stays fairly close to ~0.19% across transaction type, merchant category, device, network, bank, and state — nothing stands out as one single strong predictor on its own.
- Fraud and genuine transactions have similar amount distributions (median ₹618.5 vs ₹629).
- Fraud rate shows only mild variation by hour of day and day of week.

